# Task 4

1. Implement RNN (Recurrent Neural Network) model

    - Choose one of the tasks, either tagging or text classification or __generating__ next token.

    - Build and train RNN model that learns the chosen task.

    - Make sure to have a layer of embedding, at least one RNN layer and linear transformation before output.

    - Demonstrate that model works (on any examples).

- __(Optional)__ Implement separately embedding training module and use pre-trained embeddings instead of embedding layer before RNN.

To remember about RNN:

__Recurrence:__
Each input element at the current step is associated with the state of the network at the previous step. This state is called the hidden state and stores information about what happened before

__Feedback:__ 
One of the important features of RNN is the presence of feedback between layers. The network has loops (recurrent connections) through which information can be transmitted from the previous step to the current one.

To pytorch: 
Don't forget to reset the model loss!!

In [1]:
import os
import torch
import spacy
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [2]:
base_path = os.getcwd()
file_path = os.path.abspath(os.path.join(base_path, "..", "data", "without_chap_and_title", "eng_Anne_full_abbr.txt"))
text = open(file_path, encoding="utf-8").read()

In [ ]:
# spacy tokenization
nlp = spacy.load("en_core_web_sm")

In [4]:
def tokenize_text(text):
    doc = nlp(text)
    return [token.text.lower() for token in doc if not token.is_punct and not token.is_space]

tokens = tokenize_text(text)

In [ ]:
# сreating a vocabulary
vocab = set(tokens)
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx) 
print(vocab_size)

In [6]:
# training CBOW for pre-trained embeddings 
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)
    
    def forward(self, context_idxs):
        embeds = self.embeddings(context_idxs).mean(dim=1)
        out = self.linear(embeds)
        return out

In [7]:
# dataset to CBOW
class CBOWDataset(Dataset):
    def __init__(self, tokens, context_size):
        self.data = []
        for i in range(context_size, len(tokens) - context_size):
            context = tokens[i - context_size:i] + tokens[i + 1:i + context_size + 1]
            target = tokens[i]
            self.data.append((context, target))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        context, target = self.data[idx]
        context_idxs = torch.tensor([word2idx[w] for w in context])
        target_idx = torch.tensor(word2idx[target])
        return context_idxs, target_idx

In [ ]:
embedding_dim = 128 # default value
context_size = 2 # to fast speed
cbow_dataset = CBOWDataset(tokens, context_size)
cbow_loader = DataLoader(cbow_dataset, batch_size=64, shuffle=True) # submit data in batches of 64 examples at a time and mix them up after selection to avoid memorization

cbow_model = CBOW(vocab_size, embedding_dim)
cbow_criterion = nn.CrossEntropyLoss()
cbow_optimizer = optim.Adam(cbow_model.parameters(), lr=0.001)

In [ ]:
# training CBOW
for epoch in range(20):
    total_loss = 0 # reset the error sum
    for context_idxs, target_idx in cbow_loader:
        cbow_optimizer.zero_grad() # .zero_grad() really important !!
        output = cbow_model(context_idxs) 
        loss = cbow_criterion(output, target_idx) # error between predictions and reality
        loss.backward() # calculate gradients of errors for all parameters
        cbow_optimizer.step() # update weights 
        total_loss += loss.item() # for calculation of average loss of epoch
    print(f"[CBOW] Epoch {epoch+1}, Loss: {total_loss / len(cbow_loader):.4f}")

[CBOW] Epoch 1, Loss: 6.6365
[CBOW] Epoch 2, Loss: 5.5041
[CBOW] Epoch 3, Loss: 5.0346
[CBOW] Epoch 4, Loss: 4.6876
[CBOW] Epoch 5, Loss: 4.4043
[CBOW] Epoch 6, Loss: 4.1611
[CBOW] Epoch 7, Loss: 3.9474
[CBOW] Epoch 8, Loss: 3.7584
[CBOW] Epoch 9, Loss: 3.5912
[CBOW] Epoch 10, Loss: 3.4427
[CBOW] Epoch 11, Loss: 3.3103
[CBOW] Epoch 12, Loss: 3.1905
[CBOW] Epoch 13, Loss: 3.0812
[CBOW] Epoch 14, Loss: 2.9814
[CBOW] Epoch 15, Loss: 2.8891
[CBOW] Epoch 16, Loss: 2.8020
[CBOW] Epoch 17, Loss: 2.7223
[CBOW] Epoch 18, Loss: 2.6470
[CBOW] Epoch 19, Loss: 2.5760
[CBOW] Epoch 20, Loss: 2.5102


In [ ]:
#  use of pre-trained embeddings in RNN
pretrained_embeddings = cbow_model.embeddings.weight.data.clone() # copy so it will stay same if something goes wrong 

class TokenGeneratorRNN(nn.Module): # 
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pretrained_embeddings): 
        super(TokenGeneratorRNN, self).__init__()
        self.embeddings = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False) # layer of embeddings, freeze=False to allow network learn 
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True) # hidden_dim — how many neurons are in hidden state, 
        self.fc = nn.Linear(hidden_dim, vocab_size) # now it's probability
    
    def forward(self, x):
        embeds = self.embeddings(x)
        rnn_out, _ = self.rnn(embeds)
        output = self.fc(rnn_out[:, -1, :])  # last token in row
        return output

In [ ]:
# use of pre-trained embeddings in RNN
class TokenDataset(Dataset):
    def __init__(self, tokens, seq_length):
        self.tokens = tokens
        self.seq_length = seq_length
        self.data = self.create_sequences() # makes all sequences of words and target words
    
    def create_sequences(self):
        sequences = []
        for i in range(len(self.tokens) - self.seq_length):
            input_seq = self.tokens[i:i + self.seq_length]
            target = self.tokens[i + self.seq_length]
            sequences.append((input_seq, target))
        return sequences
    
    def __len__(self): # number of sequences
        return len(self.data)
    
    def __getitem__(self, idx): # allows you to get data by index
        input_seq, target = self.data[idx]
        input_seq_idx = torch.tensor([word2idx[word] for word in input_seq])
        target_idx = torch.tensor(word2idx[target])
        return input_seq_idx, target_idx

In [ ]:
seq_length = 2 # in word2vec was 2 and it turned out incoherent, so let's make the context bigger next time
dataset = TokenDataset(tokens, seq_length)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

hidden_dim = 256
model = TokenGeneratorRNN(vocab_size, embedding_dim, hidden_dim, pretrained_embeddings)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# RNN training
for epoch in range(20):
    total_loss = 0 #!
    for input_seq, target in dataloader:
        optimizer.zero_grad() #!
        output = model(input_seq)
        loss = loss_fn(output, target)
        loss.backward() # backpropagation
        optimizer.step() # update weqghts
        total_loss += loss.item() 
    print(f"[RNN] Epoch {epoch+1}, Loss: {total_loss / len(dataloader):.4f}")

[RNN] Epoch 1, Loss: 5.5861
[RNN] Epoch 2, Loss: 4.6713
[RNN] Epoch 3, Loss: 4.1233
[RNN] Epoch 4, Loss: 3.6646
[RNN] Epoch 5, Loss: 3.2906
[RNN] Epoch 6, Loss: 2.9715
[RNN] Epoch 7, Loss: 2.7015
[RNN] Epoch 8, Loss: 2.4685
[RNN] Epoch 9, Loss: 2.2615
[RNN] Epoch 10, Loss: 2.0821
[RNN] Epoch 11, Loss: 1.9229
[RNN] Epoch 12, Loss: 1.7843
[RNN] Epoch 13, Loss: 1.6573
[RNN] Epoch 14, Loss: 1.5455
[RNN] Epoch 15, Loss: 1.4453
[RNN] Epoch 16, Loss: 1.3576
[RNN] Epoch 17, Loss: 1.2755
[RNN] Epoch 18, Loss: 1.2024
[RNN] Epoch 19, Loss: 1.1420
[RNN] Epoch 20, Loss: 1.0809


In [ ]:
# next token generation
def generate_next_token(model, input_text, word2idx, idx2word, seq_length):
    model.eval() # evaluation 
    input_tokens = tokenize_text(input_text)
    input_idx = [word2idx[word] for word in input_tokens[-seq_length:]]  # 
    input_tensor = torch.tensor(input_idx).unsqueeze(0) # because of order of size
    
    with torch.no_grad(): # grad for eval not needed
        output = model(input_tensor) 
    predicted_idx = torch.argmax(output, dim=1).item() # we choose the most possible words
    return idx2word[predicted_idx]

In [15]:
input_text = "she was feeling"
predicted_token = generate_next_token(model, input_text, word2idx, idx2word, seq_length)
print(f"Next token after '{input_text}': {predicted_token}")

Next token after 'she was feeling': the


My conclusion: indeed the next word could have been "" - but it seems that if there were more than 5 contexts, it would have been more interesting